# Simulation analysis: spikes, rasters and recruitment

This notebook loads the results of a simulation campaign, or of a single simulation, from the
platform. It shows how to get at the spikes and plots them. It works for any circuit simulation:
- point neurons (Brian2) or detailed neurons (NEURON);
- one or several node populations, simulated (biophysical or point neuron) or virtual (only sending spikes);
- any stimuli and recordings;
- a scan over any number of parameters.

**On the platform, the data is mounted.** In the platform's sandbox, the files of your project's
simulations, and the public ones, are mounted in the file system. The notebook links to them where
they are rather than downloading them, so you can analyse many large simulation campaigns, reports
included, without copying their data to your local machine. Only files that are not mounted are downloaded: those of a
campaign of another project (open the notebook from that project to have them mounted), or
everything when the notebook runs elsewhere.

**To use it:** run the first cell, log in if a link appears under it, and pick a project in its menu.
Then choose a campaign below, and run the rest of the cells.
- The links and downloaded files go to `data/<id>/` next to this notebook, once.
- Saved figures go to `plots/<id>/`.
- The code behind the plots is in `sim_analysis.py`, next to this notebook or in a folder beside it.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

from entitysdk import LocalAssetStore
from entitysdk.client import Client
from obi_auth import get_token
from obi_notebook import get_entities
from obi_notebook.get_environment import get_environment
from obi_notebook.get_projects import get_projects

# The helper is next to this notebook, or in a folder of the supporting files extracted beside it
sys.path += [str(path.parent.resolve()) for path in Path().glob("*/sim_analysis.py")]
import sim_analysis as sa

token = get_token(environment=get_environment(), auth_mode="daf")  # click the login link if one appears
project_context = get_projects(token=token)

In [ ]:
# local_store: link files from the platform's mounted file system rather than download them
client = Client(environment=get_environment(), project_context=project_context, token_manager=token,
                local_store=LocalAssetStore(prefix="/data"))

## Select a simulation campaign

By its id or platform link, or, if `CAMPAIGN` is empty, from a table of the project's campaigns: pick
one there, then run the next cell. A single simulation's id works too, and so does a campaign of any
of your projects, or a public one.

In [ ]:
CAMPAIGN = ""

campaign_ids = [CAMPAIGN] if CAMPAIGN else get_entities.get_entities(
    "simulation-campaign", token, [], project_context=project_context, multi_select=False, page_size=100)

In [ ]:
scan = sa.load(campaign_ids[0], client)

## 1. What was simulated

Below are links to the campaign, its circuit and each simulation. The stimuli and recordings
are those of the first simulation.

When the circuit has several node populations, a table lists them. Virtual populations are not
simulated: they are inputs, such as fibres from another brain region. Only the simulated neurons
appear in the tables and rasters below. The spikes that a stimulus replays from a virtual
population show in the stimulus lanes of the raster in 3b, and shade the rasters.

In [ ]:
sa.describe(scan)

## 2. The neurons and their connectivity

### 2a. The neurons

`nodes_df = scan.nodes` gets the neurons as a pandas DataFrame: the table for working with the
data in code. It has one row per simulated neuron, with its population, its node id and a column for
each node property.

In [ ]:
nodes_df = scan.nodes
nodes_df

The **Group by** menu sets `GROUPBY_PROPERTY`: the column of `nodes_df` that groups the neurons in
the tables and plots below. The cells after it count the neurons in each group, plot where they are,
and pick a subset of them. After changing it, run the cells below again.

In [ ]:
group_menu = sa.property_menu(scan)
group_menu

In [ ]:
GROUPBY_PROPERTY = group_menu.value

# Neurons in each group, per population
nodes_df.groupby([GROUPBY_PROPERTY, "population"], observed=True).size().unstack(fill_value=0)

In [ ]:
sa.plot_positions(nodes_df, by=GROUPBY_PROPERTY)

In [ ]:
# A subset of neurons: the rows of nodes_df that meet conditions, joined with & (and) or | (or)
group = nodes_df[GROUPBY_PROPERTY].iloc[0]  # the first neuron's group, as an example
nodes_df[(nodes_df[GROUPBY_PROPERTY] == group) & (nodes_df["node_id"] < 100)]

This filtering is gathered in a widget, shown here so you can get familiar with it. It selects neurons
by property values and by id (the values of the integer column chosen in **Id column**, `node_id` by
default), and `selector.nodes` gives the selected neurons as a DataFrame. Section 3 uses the same
controls, with a **Group by** menu, to choose and group the neurons of its spike rasters and
statistics.

In [ ]:
selector = sa.NeuronSelector(scan)
selector.panel()

### 2b. The connectivity matrix

The circuit's connectivity matrices, fetched as in [Load spiking output of a circuit simulation
campaign](https://github.com/openbraininstitute/obi_platform_analysis_notebooks/tree/main/Circuit/simulation_campaign_spikes) and loaded with
[Connectome-Utilities](https://github.com/openbraininstitute/ConnectomeUtilities). On the platform
they are linked from the mounted file system. Below are the number of connections between groups,
the probability that a neuron of one group connects to a neuron of another, the whole matrix as an
image, with its neurons sorted by group, and the matrix itself, at the resolution of single neurons.

In [ ]:
matrices = sa.download_circuit_connectivity(client, scan)  # {description: file}
matrices

In [ ]:
if matrices:  # the first matrix
    connectivity = sa.load_connectivity(scan, next(iter(matrices.values())), GROUPBY_PROPERTY)
    # Connections from each group (rows) to each (columns), counted by connectivity.condense(GROUPBY_PROPERTY)
    display(sa.pathways(connectivity, GROUPBY_PROPERTY))

In [ ]:
if matrices:
    sa.plot_pathways(connectivity, GROUPBY_PROPERTY)

In [ ]:
if matrices:
    sa.plot_connectivity(connectivity, GROUPBY_PROPERTY)

In [ ]:
if matrices:
    # The connectivity at the resolution of single neurons: a sparse adjacency matrix (scipy), with a
    # row per presynaptic neuron and a column per postsynaptic one, holding each connection's value
    adjacency = connectivity.matrix
    print(f"{adjacency.shape[0]:,} neurons, {adjacency.nnz:,} connections")
    display(connectivity.vertices.head())  # the neuron of each row and column of the matrix

## 3. The spikes

`sa.load` gathered every simulation's spike file, the SONATA `spikes.h5` of its result: the node id
and the time (ms) of every spike, for each node population. Here are four ways to explore them:
- **3a. Overview:** a raster of every simulation. The panel above it selects the neurons by any number
  of node properties and by id, for every plot and table of the app below it.
- **3b. Interactive raster:** one simulation in detail, with its stimuli, recorded traces and firing
  rate.
- **3c. Statistics of the selection:** the statistic chosen in a menu, for each group in each
  simulation, and the neurons that fire.
- **3d. As a table (pandas dataframe):** one row per spike, to count and group with pandas; the
  statistics of 3c for each group; a raster drawn from the table, and histograms by group.

Section 4 saves the notebook to the project. For hands-on, technical access, look into the helper next
to this notebook, `sim_analysis.py`: `load` links or downloads the files with entitysdk,
`_read_spike_file` reads a spike file with h5py, `_read_nodes` reads the node properties with
libsonata, and `RasterApp` builds the widgets.

### 3a. Overview: every simulation

Each panel is one simulation. The selected neurons are grouped by the **Group by** property and
sorted within each group by their first spike. When the stimuli target a minority of the neurons,
those neurons sit in their own black band at the top. For a scan over two parameters, the panels form
a table: one parameter across, the other down.

Panel heights are not to a common scale: each panel spans only the selected neurons that fire in
that simulation. The grey stripes show when the stimuli push: the positive half-cycles of a
sinusoid, the pulses of a pulse train, the bursts of spikes that a stimulus replays, or the window of
any other input. Stimuli that run for the whole simulation are not shaded.

The panel at the top selects the neurons, for these rasters and for every plot and table of the app
below them:

| Control | What it does |
|---|---|
| **Select neurons by property** | Three boxes. In each, pick a property, then one or more of its values (ctrl-click, or ⌘-click on a Mac). The lists show the values of neurons that fire, with their counts. |
| **Select neurons by id** | Pick an integer column in **Id column** (`node_id` by default), then paste its values, separated by spaces, commas or new lines. Node ids can also be written `population:node_id`. |
| **Match** | *all criteria* (AND) keeps neurons that pass every filled box and the id list. *any criterion* (OR) keeps neurons that pass at least one. |
| **Group by** | Sets the property that bands and colours the rasters, and groups the statistics of 3c. It starts at `GROUPBY_PROPERTY`. |

**Save figure** writes the panels to `plots/<id>/`, together with a `.json` file describing the
selection when one is active.

Redrawing a campaign of more than 250,000 spikes takes several seconds, so there the views redraw only
when you press their **Update** button, which lights up once a setting has changed.

In [ ]:
app = sa.RasterApp(scan, group_by=GROUPBY_PROPERTY)
app.overview()

### 3b. Interactive raster

One simulation in detail: a lane per stimulus, the traces of a report, the raster of the selected
neurons grouped by a node property, and their mean firing rate. The folded panel is a copy of the
one in 3a: a change in either updates both, and everything else the app shows.

The controls under the panel set the plot:

| Control | What it does |
|---|---|
| **Simulation** | Selects the simulation. |
| **Time** | Zooms in on part of the run. |
| **Groups** | Sets how many values of the **Group by** property get their own band. The rest fold into *other*. |
| **Rank values by** | *all simulations* ranks values by how many selected neurons carry them across the scan, so each value keeps its colour in every simulation. *this simulation* ranks them within the chosen one. |
| **Sort** | Sets the order of neurons within a band. |
| **Stimulated neurons on top** | Puts the selected stimulated neurons in their own band instead of under their property value. |
| **Recorded traces**, **Report** | Add the traces of a report (for example the soma voltage) for the recorded neurons in the selection. The first report that was downloaded is shown at first. |

Above the raster, each stimulus has a lane. It shows the stimulus's time course where that has a
simple shape (step, ramp, pulses or sinusoid), and the spikes it replays for a spike replay, one row
per source neuron. Any other stimulus gets a box for its window. Within a lane, amplitudes are drawn
to scale across all simulations. The grey stripes show when the stimuli push.

**Save figure** writes the plot to `plots/<id>/`, together with a `.json` file describing the
selection when one is active.

In [ ]:
app.explorer()

### 3c. Statistics of the selection

The neurons selected in 3a, in numbers: how many are selected and how many of them fire, the
statistic chosen in the **Statistic** menu against the scanned parameter, and a table of the neurons
that fire, in the order in which they start firing across the scan. The plot has a line for each
group of the **Group by** property, coloured as in the rasters, and a dashed one for all the selected
neurons. For a single simulation, a table gives every statistic for each group. The folded panel is
another copy of the selection in 3a.

| Statistic | What it measures |
|---|---|
| **Neurons that fire**, **Fraction of the selection that fires**, **Spike count** | Counts: the selected neurons that fire at least once, as a number and as a percentage, and the number of their spikes. |
| **Mean rate of the neurons that fire**, **of all selected neurons** | Spikes per neuron per second, over the neurons that fire or over all selected ones; while the stimuli are on when they do not last the whole run. |
| **Peak population rate**, **its time** | The highest firing rate of the neurons that fire, in the time bins of the rate plot in 3b, and when it happens. |
| **Median time of the first spike** | Latency: when the neurons start firing. |
| **Mean interval between spikes** | The mean time between successive spikes of a neuron, averaged over the neurons that fire twice or more. |
| **Irregularity: CV of the intervals** | The standard deviation of a neuron's intervals over their mean, averaged over the neurons that fire three times or more: near 0 for regular firing, near 1 for random (Poisson) firing, above 1 for bursts. |
| **Synchrony: Fano factor** | The variance of the selection's spike count across time bins over its mean: near 1 when the neurons fire independently at a steady rate, larger when they fire together or the rate swings. |

The table of neurons shows the chosen statistic for each neuron when it has one (rates, the time of
the first spike, the intervals and their CV), and spike counts otherwise.

In [ ]:
app.statistics()

### 3d. As a table

This section shows a few example analyses of the results in a low-level form: a table with one row per
spike. They are meant to inspire analyses of your own; the [pandas user
guide](https://pandas.pydata.org/docs/user_guide/index.html) shows what else DataFrames can do.

- `scan.spikes()` gives one row per spike.
- `nodes_df`, from 2a, has one row per neuron: its population, its node id and the circuit's node
  properties. `properties=[...]` joins any of these properties onto the spikes.
- `scan.sims` lists the simulations in scan order. Each one has the arrays `neurons` and `times`:
  `neurons` holds the row of `nodes_df` of each spike's neuron.
- The **Statistic** menu sets `STATISTIC`, one of the statistics of 3c. `sa.group_statistics` gives
  them all for the neurons of each group, one row per group and simulation: the table after the menu
  pivots the chosen one, and the last histogram shows it for each neuron. After changing it, run the
  cells below again.

In [ ]:
spikes = scan.spikes(properties=[GROUPBY_PROPERTY])
spikes

In [ ]:
statistic_menu = sa.statistic_menu(scan)
statistic_menu

In [ ]:
STATISTIC = statistic_menu.value

# Every statistic of 3c for the neurons of each group, one row per group and simulation
group_stats = sa.group_statistics(scan, GROUPBY_PROPERTY)
# The chosen one, with a row per group and a column per simulation (one value each, so "first")
group_stats.pivot_table(index=GROUPBY_PROPERTY, columns="simulation", values=STATISTIC, aggfunc="first",
                        observed=True)

In [ ]:
# Firing rate (Hz) over the whole run of the neurons that fire most in the last simulation
sim = scan.sims[-1]
counts = spikes[spikes.simulation == sim.label].groupby(["population", "node_id", GROUPBY_PROPERTY],
                                                       observed=True).size()
(counts / (sim.tstop / 1000)).rename("rate (Hz)").sort_values(ascending=False).head(10)

In [ ]:
# A raster drawn from the table: the spikes of one simulation, one row per neuron, coloured by group
import matplotlib.pyplot as plt

sim = scan.sims[-1]  # the last simulation; scan.sims[i] for another
sim_spikes = spikes[spikes.simulation == sim.label]
row = sim_spikes.groupby([GROUPBY_PROPERTY, "population", "node_id"], observed=True).ngroup()  # neurons sorted by group
fig, ax = plt.subplots(figsize=(10, 4))
for value, group in sim_spikes.groupby(GROUPBY_PROPERTY, observed=True):
    ax.scatter(group.time_ms, row.loc[group.index], s=2, label=value)
ax.set(xlabel="time (ms)", ylabel="neuron", title=sim.label)
ax.legend(title=GROUPBY_PROPERTY, markerscale=4, loc="upper left", bbox_to_anchor=(1, 1), fontsize="small")
plt.show()

In [ ]:
# Histograms by group: when the neurons of each group fire in that simulation, one panel per group
groups = sim_spikes.groupby(GROUPBY_PROPERTY, observed=True)
largest = groups.size().nlargest(8).index  # the 8 groups with the most spikes
rows = max((len(largest) + 3) // 4, 1)  # four panels per row
fig, axes = plt.subplots(rows, 4, figsize=(12, 2.6 * rows), squeeze=False, constrained_layout=True)
for ax, value in zip(axes.flat, largest):
    ax.hist(groups.get_group(value).time_ms, bins=50, range=(0, sim.tstop))
    ax.set_title(value, fontsize="small")
for ax in axes.flat[len(largest):]:
    ax.set_visible(False)
fig.supxlabel("time (ms)")
fig.supylabel("spikes")
plt.show()

In [ ]:
# The chosen statistic for each neuron that fires in that simulation, by group: one outline per group
per_neuron = sa.neuron_statistics(scan, STATISTIC, sim, properties=[GROUPBY_PROPERTY])
shown = per_neuron.columns[-1]  # the statistic, or the spike counts for a statistic of a whole group
fig, ax = plt.subplots(figsize=(8, 4))
for value, group in per_neuron.groupby(GROUPBY_PROPERTY, observed=True):
    if value in largest:
        ax.hist(group[shown], bins=30, range=(per_neuron[shown].min(), per_neuron[shown].max()),
                histtype="step", lw=1.5, label=f"{value} ({len(group)} neurons)")
ax.set(xlabel=shown, ylabel="neurons", title=sim.label)
ax.legend(title=GROUPBY_PROPERTY, loc="upper left", bbox_to_anchor=(1, 1), fontsize="small")
plt.show()

## 4. Save this notebook to the project

To keep this analysis, save the notebook as a result notebook of the project chosen in the menu at
the top:
1. Save the notebook: ctrl+s, or ⌘+s on a Mac. The file is registered as it was last saved.
2. Run the cell below. If a login link appears, click it: the first cell's login may have expired.
3. Give the result a name and a description, and press **Register**. To replace a result you saved
   earlier under the same name, pick it in **Overwrite or create new**, then press **Update**.

In [ ]:
import os
from pathlib import Path

from obi_notebook.register_notebook_result import register_result

this_notebook = Path(os.environ.get("JPY_SESSION_NAME", "")).name
if this_notebook.endswith(".ipynb") and Path(this_notebook).exists():
    register_result._notebook_file = this_notebook  # the menu of notebook files starts at this one
token = get_token(environment=get_environment(), auth_mode="daf")
register_result.register(Client(environment=get_environment(), project_context=project_context, token_manager=token))